In [1]:
import os
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, classification_report

load_dotenv()
engine = create_engine(os.getenv("DATABASE_URL"))

query = """
SELECT *
FROM hpce.churn_model_dataset
WHERE feature_date <= '2018-08-04'
"""

df = pd.read_sql(query, engine)
print(df.shape)

(342173, 26)


In [2]:
FEATURES = [
    "daily_orders",
    "daily_spend",
    "f_30",
    "m_30",
    "f_trend_30",
    "m_trend_30",
    "recency_days"
]

X = df[FEATURES]
y = df["churn_30d"]

In [3]:
cutoff = pd.to_datetime("2018-07-01").date()

train = df[df["feature_date"] <= cutoff].copy()
test  = df[df["feature_date"] > cutoff].copy()

X_train = train[FEATURES]
y_train = train["churn_30d"]

X_test = test[FEATURES]
y_test = test["churn_30d"]

In [4]:
def show_class_balance(y, name):
    y = pd.Series(y)
    counts = y.value_counts().sort_index()
    rate = y.mean()
    print(f"\n{name} size:", len(y))
    print(f"{name} churn rate (mean of y): {rate:.4f}")
    print(f"{name} counts:")
    print(counts)

show_class_balance(y_train, "TRAIN")
show_class_balance(y_test, "TEST")


TRAIN size: 327448
TRAIN churn rate (mean of y): 0.8503
TRAIN counts:
churn_30d
0     49022
1    278426
Name: count, dtype: int64

TEST size: 14725
TEST churn rate (mean of y): 0.5836
TEST counts:
churn_30d
0    6131
1    8594
Name: count, dtype: int64


In [5]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

pipe.fit(X_train, y_train)

,steps,"[('scaler', ...), ('logreg', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0


In [6]:
probs = pipe.predict_proba(X_test)[:,1]

roc = roc_auc_score(y_test, probs)
pr  = average_precision_score(y_test, probs)

print("ROC AUC:", roc_auc_score(y_test, probs))
print("PR AUC :", average_precision_score(y_test, probs))

ROC AUC: 0.912035531193942
PR AUC : 0.9487913381417512


In [7]:
threshold = 0.50
y_pred = (probs >= threshold).astype(int)

cm = confusion_matrix(y_test, y_pred)
print("Threshold:", threshold)
print("Confusion Matrix [ [TN FP], [FN TP] ]:\n", cm)

print("\nClassification report:")
print(classification_report(y_test, y_pred, digits=4))

Threshold: 0.5
Confusion Matrix [ [TN FP], [FN TP] ]:
 [[6074   57]
 [1458 7136]]

Classification report:
              precision    recall  f1-score   support

           0     0.8064    0.9907    0.8891      6131
           1     0.9921    0.8303    0.9040      8594

    accuracy                         0.8971     14725
   macro avg     0.8993    0.9105    0.8966     14725
weighted avg     0.9148    0.8971    0.8978     14725



In [8]:
def recall_at_top_k(y_true, y_prob, k=0.10):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    n = len(y_true)
    k_n = int(np.ceil(k * n))
    idx = np.argsort(-y_prob)[:k_n]
    return y_true[idx].sum() / max(y_true.sum(), 1)

for k in [0.05, 0.10, 0.20]:
    print(f"Recall@Top{int(k*100)}%:", round(recall_at_top_k(y_test, probs, k), 4))

Recall@Top5%: 0.0851
Recall@Top10%: 0.1697
Recall@Top20%: 0.3398


In [11]:
EXPERIMENT_NAME = "hpce_churn_compare"
RUN_NAME = "LogisticRegression_30d"
MODEL_TAG = "LogisticRegression"
HORIZON_TAG = "30d"

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name=RUN_NAME):
    mlflow.set_tag("model", MODEL_TAG)
    mlflow.set_tag("horizon", HORIZON_TAG)

    mlflow.log_metric("roc_auc", float(roc))
    mlflow.log_metric("pr_auc", float(pr))

    mlflow.log_param("threshold", float(threshold))
    tn, fp, fn, tp = cm.ravel()
    mlflow.log_metric("tn", int(tn))
    mlflow.log_metric("fp", int(fp))
    mlflow.log_metric("fn", int(fn))
    mlflow.log_metric("tp", int(tp))

    for k in [0.05, 0.10, 0.20]:
        mlflow.log_metric(f"recall_at_top_{int(k*100)}pct", float(recall_at_top_k(y_test, probs, k)))

    if hasattr(pipe, "get_params"):
        mlflow.log_params(pipe.get_params())

    mlflow.sklearn.log_model(pipe, artifact_path="model")

print("✅ Logged run to MLflow for model comparison")

2026/02/01 17:54:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
D:\Thompson Rivers University\3rd Semester\Grad Project on Data Science\Final Year Design Project\hyper-personalized-customer-engine\.venv\Lib\site-packages\mlflow\models\model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
  flavor.save_model(path=local_path, mlflow_model=mlflow_model, **kwargs)


✅ Logged run to MLflow for model comparison
